# US Arb Freight Breakevens vs Freight Rates - Contract Month Evolution

This script is used to compare the US Arb Freight Breakevens with Spark30 Freight Rates for a specific contract month.

For a full explanation of how to import our Arb Breakevens or Spark30 freight data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html



### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

__This section can remain unchanged for most Spark API users.__

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.  Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/request/authentication.html

In [ ]:
# Input the path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Calling Freight & Breakevens Data

## 2.1 Fetching Breakevens data

Here we define the functions used to call the Arb Breakevens data from the Cargo Arb Breakevens API. For more information on the Cargo Arb Breakevens API, please visit the API website or download our Cargo Arb Breakevens code sample:

https://www.sparkcommodities.com/api/lng-cargo/netbacks-arb-breakevens.html

In [ ]:
# Define the function for fetching reference data
def list_netbacks(access_token):

    content = do_api_get_query(
        uri="/v1.0/netbacks/reference-data/", access_token=access_token
    )

    # retrieve release dates separately
    reldates = content["data"]["staticData"]["sparkReleases"]

    # return reference data
    refdata_dict = content["data"]

    return reldates, refdata_dict

# Fetch reference data:
reldates, refdata_dict = list_netbacks(access_token)

# formatting as a Dataframe
available_df = pd.json_normalize(refdata_dict['staticData']['fobPorts'])
available_df.head(5)

In [ ]:
from io import StringIO

# Defining function to fetch breakevens dataset

def fetch_breakevens(access_token, ticker, via=None, breakeven='freight', start=None, end=None, format='json'):

    query_params = breakeven + '/' + "?fob-port={}".format(ticker)

    if via is not None:
        query_params += "&via-point={}".format(via)
    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)

    uri = "/v1.0/netbacks/arb-breakevens/{}".format(query_params)
    print(uri)
    content = do_api_get_query(
        uri="/v1.0/netbacks/arb-breakevens/{}".format(query_params),
        access_token=access_token, format=format,
    )
    
    if format == 'json':
        my_dict = content['data']
    else:
        my_dict = content.decode('utf-8')
        my_dict = pd.read_csv(StringIO(my_dict))

    return my_dict


In [ ]:
# fetching Sabine Pass UUID from reference data
tsab = available_df[available_df["name"] == 'Sabine Pass']["uuid"].iloc[0]

# Fetching data in CSV format
break_df = fetch_breakevens(access_token, tsab, via='cogh', breakeven='freight', format='csv')

break_df['ReleaseDate'] = pd.to_datetime(break_df['ReleaseDate'])
break_df['LoadingDate'] = pd.to_datetime(break_df['LoadingDate'])

break_df.head(3)

## 2.2 Fetching Freight Prices

Here we define the functions used to call the Spark30S Freight data from the Freight Contracts API. For more information on the Freight Contracts API, please visit the API website or download our Freight Contracts code sample:

https://www.sparkcommodities.com/api/lng-freight/contracts.html

In [ ]:
# Defining function to fetch Freight dataset

def fetch_historical_price_releases(access_token, ticker, limit=4, offset=None, vessel=None):

    query_params = "?limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)
    
    # '174-2stroke' or '160-tfde'
    if vessel is not None:
        query_params += "&vessel-type={}".format(vessel)
        
    print("/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params))
        
    content = do_api_get_query(
        uri="/v1.0/contracts/{}/price-releases/{}".format(ticker, query_params),
        access_token=access_token,
    )

    my_dict = content['data']
    
    return my_dict

# Defining function to store freight prices as a Pandas DataFrame

import datetime
def fetch_prices(ticker, limit, my_vessel=None):
    my_dict_hist = fetch_historical_price_releases(access_token, ticker, limit=limit, vessel=my_vessel)
    
    release_dates = []
    period_start = []
    usd_day = []

    day_min = []
    day_max = []
    period_index = []

    for release in my_dict_hist:
            release_date = release["releaseDate"]

            data_points = release["data"][0]["dataPoints"]

            for data_point in data_points:
                
                release_dates.append(release_date)
    
                period_start_at = data_point["deliveryPeriod"]["startAt"]
                period_start.append(period_start_at)

                period_index.append(data_point["deliveryPeriod"]["name"])

                usd_day.append(data_point['derivedPrices']['usdPerDay']['spark'])
                day_min.append(data_point['derivedPrices']['usdPerDay']['sparkMin'])
                day_max.append(data_point['derivedPrices']['usdPerDay']['sparkMax'])

    ## Storing values in a Pandas DataFrame

    historical_df = pd.DataFrame({
        'ReleaseDate': release_dates,
        'Ticker': ticker,
        'PeriodStart': period_start,
        'PeriodIndex': period_index,
        'USDperday': usd_day,
        'USDperdayMax': day_max,
        'USDperdayMin': day_min})

    historical_df['USDperday'] = pd.to_numeric(historical_df['USDperday'])
    historical_df['USDperdayMax'] = pd.to_numeric(historical_df['USDperdayMax'])
    historical_df['USDperdayMin'] = pd.to_numeric(historical_df['USDperdayMin'])

    historical_df['ReleaseDate'] = pd.to_datetime(historical_df['ReleaseDate'])
    historical_df['PeriodStart'] = pd.to_datetime(historical_df['PeriodStart'])
    
    return historical_df


In [ ]:
# Using functions to fetch freight data
# Here we import Atlantic rates, with the aim to compare against US arb breakevens

spark30s = fetch_prices(ticker='spark30s', limit=len(break_df['ReleaseDate'].unique()), my_vessel='174-2stroke')
spark30s['PeriodName'] = spark30s['ReleaseDate'].dt.strftime('%b%y')

spark30ffa = fetch_prices(ticker='spark30ffa-monthly', limit=len(break_df['ReleaseDate'].unique()), my_vessel='174-2stroke')
spark30ffa['PeriodName'] = spark30ffa['PeriodStart'].dt.strftime('%b%y')

print("Columns:", spark30s.columns.tolist())

In [ ]:
spark30ffa.head(3)

# 3. Filtering for Fixing Month of Choice

In [ ]:
# Choose fixing month of interest here
month = 'Jun26'

In [ ]:
# Combining Spot and FFA datasets for month of choice
freight_df = pd.concat([
    spark30s[spark30s['PeriodName'] == month][['ReleaseDate', 'Ticker', 'PeriodName', 'USDperday', 'USDperdayMax', 'USDperdayMin']],
    spark30ffa[(spark30ffa['PeriodName'] == month) & (spark30ffa['PeriodIndex'] != 'M+0')][['ReleaseDate', 'Ticker', 'PeriodName', 'USDperday', 'USDperdayMax', 'USDperdayMin']],
])

freight_df.head(5)

In [ ]:
# Filtering breakevens data for month of choice
# PeriodName is filtered for Month+1 as breakevens data is organised by load month, not fixing month (assumption: load month = fixing month + 1)
break_df['PeriodName'] = break_df['LoadingDate'].dt.strftime('%b%y')
break_month_str = (datetime.datetime.strptime(month, '%b%y') + datetime.timedelta(days=35)).strftime('%b%y')

break_month = break_df[break_df['PeriodName'] == break_month_str]
break_month.head(5)

In [ ]:
# Merge the Freight & Breakevens datasets into one dataframe

final_df = pd.merge(
    freight_df,
    break_month[['ReleaseDate', 'FreightBreakeven']],
    left_on='ReleaseDate',
    right_on = 'ReleaseDate'
)

final_df = final_df.rename(columns={
    'USDperday': 'Spark30',
    'USDperdayMax': 'Spark30Max',
    'USDperdayMin': 'Spark30Min',
})

final_df

# 4. Plotting

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(15,7))

# Plot Freight & Breakevens datasets individually
ax.plot(final_df['ReleaseDate'],final_df['Spark30'], color = '#48C38D', linewidth=2.5, label='Spark30S (Atlantic)')
ax.plot(final_df['ReleaseDate'],final_df['FreightBreakeven'], color='#4F41F4', linewidth=2, label='US Arb [M+1] Freight Breakeven Level')

# Conditional shading to show whether Breakevens are lower/higher than current Freight rates
ax.fill_between(final_df['ReleaseDate'], final_df['Spark30'], final_df['FreightBreakeven'], \
                where=final_df['Spark30']>final_df['FreightBreakeven'], facecolor='red', interpolate=True, alpha=0.05)

ax.fill_between(final_df['ReleaseDate'], final_df['Spark30'], final_df['FreightBreakeven'], \
                where=final_df['Spark30']<final_df['FreightBreakeven'], facecolor='green', interpolate=True, alpha=0.05)

sns.despine(left=True, bottom=True)
plt.grid(True)
plt.legend()